# 🦑 ahab - Python and CLI Demo

In [2]:
import os
import json
from uuid import uuid4

## Set environment variables
os.environ["AHAB_API_KEY"] = "your_ahab_api_key"
os.environ["AHAB_API_URL"] = "https://<your_ahab_api>.azurewebsites.net" ## without /api at the end

## Python Example

In [3]:
from ahab import list_jobs, get_job, submit_job, update_job

In [4]:
## Submit using the ahab Python client
response = submit_job(
    job_type="rnaseq",
    metadata={
        "project": "nextflow-samples",
        "sample": "ggal",
    },
    inputs={
        "transcriptome": "/mnt/datalake/references/rnaseq-nf/ggal_1_48850000_49020000.Ggal71.500bpflank.fa",
        "reads": "/mnt/datalake/samples/nf_ggal/ggal_liver_{1,2}.fq",
        "outdir": f"/mnt/datalake/samples/nf_ggal/liver_results_{uuid4()}"
    },
)

print(response)

{'message': 'Job created successfully.', 'id': 'f3488969-f46d-47ef-a332-dffc124d2d21', 'job_type': 'rnaseq'}


In [5]:
## Submit using a dictionary
response = submit_job(body={
        "job_type": "rnaseq",
        "metadata": {
            "project": "nextflow-samples",
            "sample": "ggal"
        },
        "inputs": {
            "transcriptome": "/mnt/datalake/references/rnaseq-nf/ggal_1_48850000_49020000.Ggal71.500bpflank.fa",
            "reads": "/mnt/datalake/samples/nf_ggal/ggal_liver_{1,2}.fq",
            "outdir": f"/mnt/datalake/samples/nf_ggal/liver_results_{uuid4()}"
        }
    }
)

print(response)

{'message': 'Job created successfully.', 'id': 'a35d696d-6b71-40d2-ab53-84b611a8bc0a', 'job_type': 'rnaseq'}


In [6]:
## Submit using a JSON file
response = submit_job(json_file="examples/rnaseq_nf_ggal.json")

print(response)

{'message': 'Job created successfully.', 'id': 'fdc8dd6e-98e5-48f7-a030-519937c58f1d', 'job_type': 'rnaseq'}


In [11]:
## List all jobs without changing status.
jobs = list_jobs()

print(json.dumps(jobs, indent=4))

{
    "message": "5 job(s) found.",
    "jobs": [
        {
            "id": "f3488969-f46d-47ef-a332-dffc124d2d21",
            "job_type": "rnaseq",
            "current_status": "COMPLETE",
            "status_history": {
                "2026-11-14T17:15:33.380692": "UNSCHEDULED",
                "2026-11-14T17:16:00.338330": "PENDING",
                "2026-11-14T17:16:57.115942": "STARTED",
                "2026-11-14T17:17:03.713762": "COMPLETE"
            },
            "metadata": {
                "project": "nextflow-samples",
                "sample": "ggal"
            },
            "inputs": {
                "transcriptome": "/mnt/datalake/references/rnaseq-nf/ggal_1_48850000_49020000.Ggal71.500bpflank.fa",
                "reads": "/mnt/datalake/samples/nf_ggal/ggal_liver_{1,2}.fq",
                "outdir": "/mnt/datalake/samples/nf_ggal/liver_results_3658f732-6163-4d55-9508-33cd6c61b48f"
            },
            "image": "tuplexyz/ahab-rnaseq-nf:v1.2.3",
        

In [8]:
## Worker claim: returns one pending job and changes its status to STARTED.
job = get_job(job_type="rnaseq")

update_job(
    id=job["id"],
    status="COMPLETE",
    body={"outputs": {"bam": "/mnt/results/sample.bam"}},
)

{'message': 'Job updated successfully.',
 'id': 'f3488969-f46d-47ef-a332-dffc124d2d21',
 'status': 'COMPLETE'}

## CLI Example

In [ ]:
%%bash
ahab job submit \
  --job-type rnaseq \
  --metadata project=nextflow-samples \
  --metadata sample=ggal \
  --input transcriptome=/mnt/datalake/references/rnaseq-nf/ggal_1_48850000_49020000.Ggal71.500bpflank.fa \
  --input reads=/mnt/datalake/samples/nf_ggal/ggal_liver_{1,2}.fq \
  --input outdir=/mnt/datalake/samples/nf_ggal/liver_results_45678

In [ ]:
## Claim a pending job for a worker (the API changes it to STARTED)
!ahab job get --job_type=rnaseq

In [ ]:
## Change the status of a job
!ahab job update --id=fb273f3b-9665-4f9d-978c-9f6cb9f1cc19 --status=COMPLETE

In [6]:
## List every job without claiming or changing its status
!ahab job list

In [ ]:
## Complete JSON can instead be provided inline or read from a file.
# ahab job submit --body='{"job_type":"rnaseq","metadata":{"sample":"sample_001"},"inputs":{"fastq_1":"/mnt/data/file_1.fastq.gz"}}'
!ahab job submit --file=examples/rnaseq_nf_ggal.json